# PFC · Avaliação dos modelos em GPU de nuvem (Colab)

Roda o **mesmo** `pfc-avaliar` do repositório, com o **mesmo** Ollama, os **mesmos** modelos e a **mesma** quantização — numa T4 de 16 GB, onde todos cabem inteiros na VRAM.

O que muda em relação à estação do Cap. 3 é só o hardware, e isso fica registrado no `manifesto.json` de cada modelo (`nvidia-smi`, versões, hash do dataset). No texto, o ambiente é descrito **como ele é**.

**Antes de começar:** `Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU`. Depois `Ambiente de execução → Executar tudo` (ou célula por célula). A célula 3 pede o arquivo `pfc_busca_colab.zip`. Deixe a aba aberta; a rodada leva cerca de 1 h 30 min. Se o Colab perguntar se você ainda está aí, confirme.

> Nenhum serviço de LLM em nuvem é usado: o Ollama roda dentro desta máquina virtual, com os pesos baixados aqui. Só a GPU é alugada.

**Use sempre um ambiente de execução novo** (`Ambiente de execução → Desconectar e excluir ambiente de execução`) e o zip mais recente: a célula 3 confere o hash do dataset e se recusa a rodar com um pacote antigo.

**Se a sessão cair no meio:** reabra, `Executar tudo` de novo e reenvie o zip se ele pedir. O avaliador pula o que já foi feito — mas só se a pasta `results/` ainda existir; se o Colab reciclar a máquina, a rodada recomeça do zero.

In [ ]:
# 1) Configuração do experimento — NÃO altere depois de começar
MODELOS = ['qwen3:4b-instruct-2507-q4_K_M', 'gemma4:e4b-it-qat', 'gemma4:e2b-it-qat', 'mistral-nemo:12b']
HOJE = '2026-09-14'     # data de referência fixa: injetada no prompt e usada no gabarito (a mesma da validação local)
REPETICOES = 3
DATASET_SHA256 = '07bf424a40d68392cad4a190c334b3b6144676d2927a75ae6b0b5328abf0ba42'   # gravado por scripts/empacotar_colab.py: o pacote precisa trazer ESTE dataset
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2) Instala o Ollama (precisa de zstd) e sobe o servidor
!apt-get -qq update > /dev/null 2>&1 && apt-get -qq install -y zstd pciutils > /dev/null 2>&1 && echo "zstd instalado"
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -3

import os, shutil, subprocess, time, json, urllib.request
assert shutil.which('ollama'), 'o Ollama NÃO foi instalado — veja as mensagens acima'
servidor = subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT,
                            env=dict(os.environ, OLLAMA_KEEP_ALIVE='30m'))
for _ in range(40):
    try:
        v = json.load(urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=3))['version']
        print('Ollama', v, 'no ar')
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('o servidor do Ollama não subiu — veja /content/ollama.log')

In [ ]:
# 3) Envia o repositório (pfc_busca_colab.zip), confere o dataset e instala o pacote
import os, hashlib, shutil, datetime
from google.colab import files

def sha_dataset():
    p = '/content/pfc_busca/data/dataset.json'
    return hashlib.sha256(open(p, 'rb').read()).hexdigest() if os.path.exists(p) else None

if os.path.exists('/content/pfc_busca') and sha_dataset() != DATASET_SHA256:
    antigo = '/content/pfc_busca_anterior_' + datetime.datetime.now().strftime('%H%M%S')
    shutil.move('/content/pfc_busca', antigo)
    print('pacote anterior movido para', antigo, '(não será usado)')
if not os.path.exists('/content/pfc_busca/pyproject.toml'):
    enviado = files.upload()
    nome = next(iter(enviado))
    !unzip -q -o "$nome" -d /content
assert sha_dataset() == DATASET_SHA256, (
    'O zip enviado NÃO é o pacote atual (o hash do dataset não confere). Baixe de novo dist/pfc_busca_colab.zip.')
print('dataset conferido:', DATASET_SHA256[:12])
%cd /content/pfc_busca
!pip install -q -e . 2>&1 | tail -2
!python -c "from pfc_busca.evaluation.dataset_builder import carregar_dataset; print(len(carregar_dataset()), 'consultas no dataset')"


In [ ]:
# 4) Baixa os quatro modelos (~19 GB; 5-10 min)
for m in MODELOS:
    print(m, end=': ', flush=True)
    !ollama pull {m} 2>&1 | tail -1
!ollama list

In [ ]:
# 5) Rodada F6: 310 consultas x 3 repetições por modelo (cerca de 1 h 30 min no total)
#    Mostra uma linha a cada 50 consultas e toda falha de infraestrutura (×).
import datetime, re, subprocess

def rodar(modelo):
    cmd = ['pfc-avaliar', '--modelo', modelo, '--repeticoes', str(REPETICOES), '--hoje', HOJE, '--sem-sql']
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env=dict(os.environ, COLUMNS='110'))
    n = 0
    for linha in proc.stdout:
        linha = linha.rstrip()
        if re.match(r'^[✓·×] r\d', linha):
            n += 1
            if linha.startswith('×') or n % 50 == 0:
                print(f'   [{n}] {linha}', flush=True)
        elif linha.strip() and not linha.startswith('avaliando'):
            print(linha, flush=True)
    return proc.wait()

for m in MODELOS:
    print(f'\n== {m} == início {datetime.datetime.now():%H:%M}', flush=True)
    codigo = rodar(m)
    print(f'   saída {codigo} às {datetime.datetime.now():%H:%M}', flush=True)
    subprocess.run(['ollama', 'stop', m], capture_output=True)

In [ ]:
# 6) Consolida (tabelas do Cap. 5, figuras, estatística) e baixa tudo
!pfc-relatorio --modelos {','.join(MODELOS)}
!cd /content && rm -f resultados_colab.zip && zip -q -r resultados_colab.zip pfc_busca/results && ls -la resultados_colab.zip
files.download('/content/resultados_colab.zip')

## De volta ao PC

Descompacte `resultados_colab.zip` na pasta que contém `pfc_busca/` (os registros vão para `pfc_busca/results/`), confira as rodadas com `python -m pfc_busca.evaluation.conferencia` e regenere as tabelas e figuras do texto com `python scripts/gerar_tabelas_texto.py --texto texto`.